# Task Queue & Multi-Provider Dispatch

A platform without a queue is just a remote shell. The queue is what separates "run this job" from "run this job when a suitable machine is available, retry if it fails, and keep a record of everything." Without it, two jobs submitted simultaneously might collide on the same machine, a transient network error would silently drop a run, and there would be no way to inspect what happened after the fact.

This notebook builds the task queue from first principles using `asyncio`, adds a lifecycle state machine that tracks every job through its transitions, and wraps it all with a provider selection layer that routes each job to the best available machine across multiple backends.

## Queue Fundamentals

The Python standard library provides two async queue types that cover nearly every use case:

- `asyncio.Queue` — FIFO ordering; items are consumed in submission order.
- `asyncio.PriorityQueue` — items are `(priority, item)` tuples; lower $p$ means higher priority.

Both support `put(item)` to enqueue (async; blocks if `maxsize` is set and the queue is full) and `get()` to dequeue (async; blocks if empty). `put_nowait` and `get_nowait` raise immediately rather than blocking.

**Why a priority queue.** Most ML workloads are batch jobs that can wait, but some are urgent: a hyperparameter sweep that a researcher is watching live, or a monitoring job that should pre-empt long-running training runs. A `PriorityQueue` with priorities `1` (urgent) and `3` (batch) gives a simple two-tier system without the complexity of a separate queue per priority level.

**`maxsize` and backpressure.** Setting `maxsize=N` causes `put()` to block when the queue has $N$ items, which propagates backpressure to the producer: if workers are busy, the API route that accepts job submissions will wait rather than accepting an unbounded number of jobs into memory. For the NBX platform, an unbounded queue is usually acceptable — jobs are small objects — but `maxsize` is useful in testing to force backpressure scenarios.

`asyncio.Queue` and `asyncio.PriorityQueue` are in-process only. A multi-server deployment requires a distributed queue (Redis, SQS, or RabbitMQ). For single-process deployments, the standard library queue is sufficient and has zero operational overhead.

In [ ]:
import asyncio


async def producer(q: asyncio.Queue, n: int):
    for i in range(n):
        await q.put(f"job-{i}")
        print(f"  enqueued job-{i}")
        await asyncio.sleep(0.05)


async def consumer(q: asyncio.Queue, worker_id: int):
    while True:
        item = await q.get()
        print(f"  [worker-{worker_id}] processing {item}")
        await asyncio.sleep(0.1)  # simulate work
        q.task_done()


async def run_fifo_demo():
    q    = asyncio.Queue()
    prod = asyncio.create_task(producer(q, 5))
    work = asyncio.create_task(consumer(q, worker_id=0))
    await prod
    await q.join()
    work.cancel()


asyncio.run(run_fifo_demo())

Now with `PriorityQueue` — lower number jumps the queue:

In [ ]:
async def run_priority_demo():
    q: asyncio.PriorityQueue = asyncio.PriorityQueue()

    items = [
        (3, "batch-job-A"),
        (1, "urgent-job-B"),
        (3, "batch-job-C"),
        (2, "normal-job-D"),
        (1, "urgent-job-E"),
    ]
    for item in items:
        await q.put(item)

    print("Processing order:")
    while not q.empty():
        priority, name = await q.get()
        print(f"  p={priority}  {name}")
        q.task_done()


asyncio.run(run_priority_demo())

## Job Lifecycle FSM

Every job passes through a well-defined set of states. Modeling these states explicitly — and enforcing legal transitions — prevents a large class of bugs where a job ends up in an impossible state such as transitioning from `done` back to `running`.

The finite state machine has six states and the following legal transitions:

$$
\text{pending} \to \text{queued} \to \text{running} \to \begin{cases} \text{done} \\ \text{failed} \end{cases}
$$

Any non-terminal state can also transition to `cancelled` via an explicit cancellation request. We define the enum, the record model, and the transition validator:

In [ ]:
from __future__ import annotations

import uuid
from datetime import datetime, timezone
from enum import Enum
from typing import Any

from pydantic import BaseModel, Field


class JobStatus(str, Enum):
    pending   = "pending"
    queued    = "queued"
    running   = "running"
    done      = "done"
    failed    = "failed"
    cancelled = "cancelled"


TERMINAL = {JobStatus.done, JobStatus.failed, JobStatus.cancelled}

LEGAL_TRANSITIONS: dict[JobStatus, set[JobStatus]] = {
    JobStatus.pending:   {JobStatus.queued,   JobStatus.cancelled},
    JobStatus.queued:    {JobStatus.running,  JobStatus.cancelled},
    JobStatus.running:   {JobStatus.done,     JobStatus.failed, JobStatus.cancelled},
    JobStatus.done:      set(),
    JobStatus.failed:    set(),
    JobStatus.cancelled: set(),
}


class JobSpec(BaseModel):
    id:         str = Field(default_factory=lambda: uuid.uuid4().hex[:8])
    entrypoint: str
    params:     dict[str, Any] = {}


class JobRecord(BaseModel):
    id:          str              = Field(default_factory=lambda: uuid.uuid4().hex[:8])
    spec:        JobSpec
    status:      JobStatus        = JobStatus.pending
    priority:    int              = 3
    created_at:  datetime         = Field(default_factory=lambda: datetime.now(timezone.utc))
    started_at:  datetime | None  = None
    finished_at: datetime | None  = None
    error:       str | None       = None


def transition(record: JobRecord, new_status: JobStatus) -> JobRecord:
    """Return an updated record after validating the status transition."""
    allowed = LEGAL_TRANSITIONS[record.status]
    if new_status not in allowed:
        raise ValueError(
            f"Illegal transition: {record.status} -> {new_status}. "
            f"Allowed: {allowed or 'terminal state'}"
        )
    updates: dict[str, Any] = {"status": new_status}
    now = datetime.now(timezone.utc)
    if new_status == JobStatus.running:
        updates["started_at"] = now
    if new_status in TERMINAL:
        updates["finished_at"] = now
    return record.model_copy(update=updates)

Verifying that legal transitions succeed and illegal ones raise:

In [ ]:
spec = JobSpec(entrypoint="train.py")
rec  = JobRecord(spec=spec)

rec = transition(rec, JobStatus.queued)
rec = transition(rec, JobStatus.running)
rec = transition(rec, JobStatus.done)
print(f"Final status: {rec.status}  finished_at set: {rec.finished_at is not None}")

try:
    transition(rec, JobStatus.running)
except ValueError as e:
    print(f"Caught: {e}")

## The Queue Worker

`QueueWorker` is an async loop that pulls `JobRecord` objects from a shared `asyncio.PriorityQueue`, calls the dispatcher, and updates job status on completion. Multiple workers run concurrently via `asyncio.gather`, giving us configurable parallelism. We use an `asyncio.Event` as the stop signal — when set, workers finish any in-flight jobs and exit cleanly.

A mock dispatcher that simulates execution with a 20% random failure rate:

In [ ]:
import random


class MockDispatcher:
    """Pretends to run a job: sleeps, then fails 20% of the time."""

    async def dispatch(self, record: JobRecord) -> None:
        await asyncio.sleep(random.uniform(0.05, 0.2))
        if random.random() < 0.2:
            raise RuntimeError(f"Simulated failure for {record.id}")

The full `QueueWorker` class:

In [ ]:
job_store: dict[str, JobRecord] = {}


class QueueWorker:
    def __init__(
        self,
        queue:      asyncio.PriorityQueue,
        dispatcher: MockDispatcher,
        worker_id:  int,
        stop:       asyncio.Event,
    ):
        self.queue      = queue
        self.dispatcher = dispatcher
        self.worker_id  = worker_id
        self.stop       = stop

    async def run(self) -> None:
        while not self.stop.is_set() or not self.queue.empty():
            try:
                priority, record = await asyncio.wait_for(
                    self.queue.get(), timeout=0.1
                )
            except asyncio.TimeoutError:
                continue

            # Skip jobs cancelled via the API before being picked up
            if job_store.get(record.id, record).status == JobStatus.cancelled:
                self.queue.task_done()
                print(f"  [W{self.worker_id}] skip   {record.id} (cancelled)")
                continue

            print(f"  [W{self.worker_id}] start  {record.id} (p={priority})")
            record = transition(record, JobStatus.running)
            job_store[record.id] = record

            try:
                await self.dispatcher.dispatch(record)
                record = transition(record, JobStatus.done)
                print(f"  [W{self.worker_id}] done   {record.id}")
            except Exception as exc:
                record = record.model_copy(update={"error": str(exc)})
                record = transition(record, JobStatus.failed)
                print(f"  [W{self.worker_id}] FAILED {record.id}: {exc}")
            finally:
                job_store[record.id] = record
                self.queue.task_done()

Spinning up 2 workers and submitting 8 jobs to watch concurrent processing:

In [ ]:
async def run_worker_demo():
    job_store.clear()
    q    = asyncio.PriorityQueue()
    stop = asyncio.Event()
    disp = MockDispatcher()

    for i in range(8):
        p    = 1 if i % 3 == 0 else 3
        spec = JobSpec(entrypoint=f"job_{i}.py")
        rec  = JobRecord(spec=spec, priority=p)
        rec  = transition(rec, JobStatus.queued)
        job_store[rec.id] = rec
        await q.put((p, rec))

    workers = [QueueWorker(q, disp, wid, stop) for wid in range(2)]

    async def stopper():
        await q.join()
        stop.set()

    await asyncio.gather(*[w.run() for w in workers], stopper())

    statuses = [r.status for r in job_store.values()]
    print(f"\ndone={statuses.count(JobStatus.done)}  failed={statuses.count(JobStatus.failed)}")


asyncio.run(run_worker_demo())

## Provider Selection Strategy

The `Dispatcher` sits between the worker and the actual execution layer. It queries the machine registry for suitable machines and selects one based on a configurable strategy:

- **`round_robin`** — rotate through online machines sequentially; simple and fair under homogeneous load.
- **`least_loaded`** — pick the machine with the lowest current CPU load from the metrics cache; optimal when machines have very different utilization.
- **`tagged`** — filter machines by a tag specified in `job_spec.params["NBX_TARGET_TAG"]`; useful for routing GPU jobs to GPU-only machines.

Defining the strategy enum and the dispatcher:

In [ ]:
import itertools
from dataclasses import dataclass, field as dc_field


class DispatchStrategy(str, Enum):
    round_robin  = "round_robin"
    least_loaded = "least_loaded"
    tagged       = "tagged"


@dataclass
class MockMachine:
    id:       str
    name:     str
    status:   str        = "online"
    tags:     list[str]  = dc_field(default_factory=list)
    cpu_load: float      = 1.0


class Dispatcher:
    def __init__(
        self,
        machines: list[MockMachine],
        strategy: DispatchStrategy = DispatchStrategy.round_robin,
    ):
        self.strategy  = strategy
        self._machines = machines
        self._rr_iter  = itertools.cycle(
            m for m in machines if m.status == "online"
        )

    def _online(self, tag: str | None = None) -> list[MockMachine]:
        ms = [m for m in self._machines if m.status == "online"]
        if tag:
            ms = [m for m in ms if tag in m.tags]
        return ms

    def select(self, record: JobRecord) -> MockMachine | None:
        if self.strategy == DispatchStrategy.tagged:
            tag        = record.spec.params.get("NBX_TARGET_TAG")
            candidates = self._online(tag)
        else:
            candidates = self._online()

        if not candidates:
            return None

        if self.strategy == DispatchStrategy.round_robin:
            return next(self._rr_iter)

        if self.strategy == DispatchStrategy.least_loaded:
            return min(candidates, key=lambda m: m.cpu_load)

        return candidates[0]

Unit-testing all three strategies against a mock registry with 3 machines:

In [ ]:
mock_machines = [
    MockMachine(id="m1", name="gpu-01", tags=["gpu", "a100"], cpu_load=3.5),
    MockMachine(id="m2", name="gpu-02", tags=["gpu", "rtx"],  cpu_load=0.4),
    MockMachine(id="m3", name="cpu-01", tags=["cpu"],          cpu_load=1.2),
]
recs = [JobRecord(spec=JobSpec(entrypoint=f"j{i}.py")) for i in range(6)]

# Round-robin: cycles through all three machines
rr = Dispatcher(mock_machines, DispatchStrategy.round_robin)
print("Round-robin:", [rr.select(r).name for r in recs])

# Least-loaded: always picks gpu-02 (cpu_load=0.4)
ll = Dispatcher(mock_machines, DispatchStrategy.least_loaded)
print("Least-loaded:", [ll.select(r).name for r in recs[:3]])

# Tagged: only GPU-tagged machines
tg      = Dispatcher(mock_machines, DispatchStrategy.tagged)
gpu_rec = JobRecord(spec=JobSpec(entrypoint="train.py", params={"NBX_TARGET_TAG": "gpu"}))
sel     = tg.select(gpu_rec)
print(f"Tagged (gpu): {sel.name}  tags={sel.tags}")

## Concurrency Control

Without concurrency limits, the dispatcher will happily route 50 simultaneous jobs to a machine with 8 CPUs, each process competing for the same physical cores and causing every job to run more slowly than if they had run sequentially. `asyncio.Semaphore` caps the number of inflight jobs per machine — any submission beyond the limit awaits until a slot is released.

**Setting `MAX_CONCURRENT`.** A machine with $k$ GPUs should typically allow $k$ simultaneous GPU-bound training jobs and $2k$ CPU-bound preprocessing jobs. Oversubscribing by a small factor ($ \approx 1.5\times$) exploits I/O wait time — data loading, checkpoint saves — without causing memory pressure. The conservative default of `MAX_CONCURRENT = 2` is a safe starting point; raise it only after confirming GPU utilization stays below 80%.

**Per-machine semaphores.** The `machine_semaphores` dict gives each machine its own independent limit. A slow job on `gpu-01` does not block submissions to `gpu-02`. The dict is created lazily on first access and lives for the process lifetime — no external state is needed.

In [ ]:
MAX_CONCURRENT = 2
machine_semaphores: dict[str, asyncio.Semaphore] = {}


def get_semaphore(machine_id: str) -> asyncio.Semaphore:
    if machine_id not in machine_semaphores:
        machine_semaphores[machine_id] = asyncio.Semaphore(MAX_CONCURRENT)
    return machine_semaphores[machine_id]

Demonstrating that 5 simultaneous jobs to one machine with `MAX_CONCURRENT=2` run at most 2 at a time:

In [ ]:
active_count = 0
max_observed = 0


async def run_job(job_id: str, machine_id: str):
    global active_count, max_observed
    sem = get_semaphore(machine_id)
    async with sem:
        active_count += 1
        max_observed  = max(max_observed, active_count)
        print(f"  {job_id} started  (active={active_count})")
        await asyncio.sleep(0.15)
        print(f"  {job_id} done     (active={active_count})")
        active_count -= 1


async def concurrency_demo():
    global active_count, max_observed
    active_count = 0
    max_observed = 0
    machine_semaphores.clear()
    await asyncio.gather(*[run_job(f"job-{i}", "m1") for i in range(5)])
    print(f"\nmax concurrent observed: {max_observed}  (limit={MAX_CONCURRENT})")


asyncio.run(concurrency_demo())

:::{.callout-note}
`asyncio.Semaphore` is a single-process primitive. In a multi-worker deployment — multiple API server processes — you need a distributed lock such as Redis `SETNX` to enforce the same limit across all processes. For single-process deployments the `asyncio.Semaphore` is sufficient and has zero overhead.

:::

## FastAPI Job Endpoints

The job router exposes four endpoints covering submission, listing, retrieval, and cancellation. The queue and job store are module-level singletons — in production these are injected via FastAPI's dependency system.

Defining the router:

In [ ]:
from fastapi import APIRouter, FastAPI, HTTPException
from fastapi import Query as FQuery

jobs_store: dict[str, JobRecord] = {}
jobs_queue: asyncio.PriorityQueue = asyncio.PriorityQueue()

jobs_router = APIRouter(prefix="/jobs", tags=["jobs"])


@jobs_router.post("/", response_model=JobRecord, status_code=201)
async def submit_job(
    spec: JobSpec, priority: int = FQuery(default=3, ge=1, le=5)
) -> JobRecord:
    record = JobRecord(spec=spec, priority=priority)
    record = transition(record, JobStatus.queued)
    jobs_store[record.id] = record
    await jobs_queue.put((priority, record))
    return record


@jobs_router.get("/", response_model=list[JobRecord])
async def list_jobs(
    status: JobStatus | None = FQuery(default=None)
) -> list[JobRecord]:
    records = list(jobs_store.values())
    if status is not None:
        records = [r for r in records if r.status == status]
    return records


@jobs_router.get("/{job_id}", response_model=JobRecord)
async def get_job(job_id: str) -> JobRecord:
    if job_id not in jobs_store:
        raise HTTPException(status_code=404, detail="Job not found")
    return jobs_store[job_id]


@jobs_router.delete("/{job_id}", status_code=204)
async def cancel_job(job_id: str) -> None:
    if job_id not in jobs_store:
        raise HTTPException(status_code=404, detail="Job not found")
    record = jobs_store[job_id]
    if record.status in TERMINAL:
        raise HTTPException(status_code=409, detail=f"Job already {record.status}")
    jobs_store[job_id] = transition(record, JobStatus.cancelled)

Exercising the router with an httpx async client — submit 3 jobs, list by status, then cancel one:

In [ ]:
import httpx
from httpx import ASGITransport

job_app = FastAPI()
job_app.include_router(jobs_router)


async def jobs_demo():
    jobs_store.clear()
    transport = ASGITransport(app=job_app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as c:

        ids = []
        for i in range(3):
            r = await c.post(
                "/jobs/",
                json={"entrypoint": f"train_{i}.py"},
                params={"priority": 3},
            )
            ids.append(r.json()["id"])
            print(f"Submitted: {r.json()['id']}  status={r.json()['status']}")

        r = await c.get("/jobs/", params={"status": "queued"})
        print(f"Queued: {len(r.json())}")

        r = await c.delete(f"/jobs/{ids[0]}")
        print(f"Cancel HTTP {r.status_code}")

        r = await c.get(f"/jobs/{ids[0]}")
        print(f"Status after cancel: {r.json()['status']}")


asyncio.run(jobs_demo())

:::{.callout-important}
Cancelling a queued job via the API marks it `cancelled` in the store, but the `JobRecord` already sitting in the `asyncio.PriorityQueue` is unaffected. Workers must re-read `jobs_store[record.id].status` after dequeuing and skip cancelled items — the `QueueWorker.run` implementation above does exactly this. Omitting that check causes cancelled jobs to execute anyway.

:::

---

■